# RF-DETR Seg Small Optimization v1 (4GB VRAM)

This notebook is the transformer-focused training and evaluation pipeline.

It supports:
- constrained training sweep (hardware-aware)
- confidence calibration per trained run
- per-instance and global extraction reports
- IoU visual overlays
- inference-time studies (latency, throughput, peak VRAM)


In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path
from typing import Dict, List, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from rfdetr import RFDETRSegSmall

In [ ]:
def discover_repo_root(start: Path | None = None) -> Path:
    p = (start or Path.cwd()).resolve()
    while p != p.parent:
        if (p / '.git').exists() and (p / 'v2_rf_detr').exists():
            return p
        p = p.parent
    raise RuntimeError('Could not find repo root')


REPO_ROOT = discover_repo_root()
PROJECT_ROOT = REPO_ROOT / 'v2_rfdetr_opt_v1'
ARTIFACTS_ROOT = PROJECT_ROOT / 'artifacts'
RUNS_ROOT = ARTIFACTS_ROOT / 'runs'
REPORTS_ROOT = ARTIFACTS_ROOT / 'reports'
VIZ_ROOT = ARTIFACTS_ROOT / 'iou_viz'
BENCH_ROOT = ARTIFACTS_ROOT / 'benchmarks'
for p in [RUNS_ROOT, REPORTS_ROOT, VIZ_ROOT, BENCH_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# Dataset lives outside repo in current project setup.
DATASET_DIR = (REPO_ROOT.parent.parent / 'dataset_v2_coco_rf_detr').resolve()

IMG_W = 640
IMG_H = 640
IOU_THRESHOLD = 0.5
AREA_FACTOR = (50 / 72) ** 2
CONF_CANDIDATES = [round(x, 2) for x in np.arange(0.20, 0.71, 0.02)]

# Toggle this to False to skip expensive training and evaluate existing run folders.
RUN_TRAINING = True

# 4GB-VRAM-oriented constrained sweep.
RUN_MATRIX = [
    {
        'run_name': 'seg_small_optA_r560_acc16_lr1e4',
        'epochs': 120,
        'batch_size': 1,
        'grad_accum_steps': 16,
        'lr': 1e-4,
        'resolution': 560,
        'gradient_checkpointing': True,
        'early_stopping_patience': 30,
    },
    {
        'run_name': 'seg_small_optB_r504_acc16_lr1e4',
        'epochs': 120,
        'batch_size': 1,
        'grad_accum_steps': 16,
        'lr': 1e-4,
        'resolution': 504,
        'gradient_checkpointing': True,
        'early_stopping_patience': 30,
    },
    {
        'run_name': 'seg_small_optC_r560_acc20_lr1e4',
        'epochs': 120,
        'batch_size': 1,
        'grad_accum_steps': 20,
        'lr': 1e-4,
        'resolution': 560,
        'gradient_checkpointing': True,
        'early_stopping_patience': 30,
    },
]

print('REPO_ROOT:   ', REPO_ROOT)
print('DATASET_DIR: ', DATASET_DIR)
print('ARTIFACTS:   ', ARTIFACTS_ROOT)
print('RUN_TRAINING:', RUN_TRAINING)
print('RUNS:', [r['run_name'] for r in RUN_MATRIX])

In [ ]:
def summarize_split(split: str) -> None:
    ann_path = DATASET_DIR / split / '_annotations.coco.json'
    if not ann_path.exists():
        print(f'[{split}] missing: {ann_path}')
        return

    data = json.loads(ann_path.read_text(encoding='utf-8'))
    anns = data.get('annotations', [])
    seg_n = sum(1 for a in anns if a.get('segmentation'))
    cat_counts = {}
    for a in anns:
        cid = int(a.get('category_id', -1))
        cat_counts[cid] = cat_counts.get(cid, 0) + 1
    cats = {int(c.get('id', -1)): c.get('name', '') for c in data.get('categories', [])}

    print(f'[{split}] images: {len(data.get("images", []))}  annotations: {len(anns)}  with segmentation: {seg_n}')
    print(f'[{split}] category_id counts: {cat_counts}')
    print(f'[{split}] categories: {cats}')


summarize_split('train')
summarize_split('valid')

In [ ]:
def run_training(cfg: Dict) -> Path:
    output_dir = RUNS_ROOT / cfg['run_name']
    output_dir.mkdir(parents=True, exist_ok=True)

    model = RFDETRSegSmall()
    model.train(
        dataset_dir=str(DATASET_DIR),
        output_dir=str(output_dir),
        epochs=int(cfg['epochs']),
        batch_size=int(cfg['batch_size']),
        grad_accum_steps=int(cfg['grad_accum_steps']),
        lr=float(cfg['lr']),
        gradient_checkpointing=bool(cfg['gradient_checkpointing']),
        resolution=int(cfg['resolution']),
        early_stopping=True,
        early_stopping_patience=int(cfg['early_stopping_patience']),
        progress_bar=True,
        seed=0,
    )
    return output_dir


trained_run_dirs: Dict[str, Path] = {}
if RUN_TRAINING:
    for cfg in RUN_MATRIX:
        print(f"\n=== Training {cfg['run_name']} ===")
        out = run_training(cfg)
        trained_run_dirs[cfg['run_name']] = out
        print('Finished:', out)
else:
    for cfg in RUN_MATRIX:
        trained_run_dirs[cfg['run_name']] = RUNS_ROOT / cfg['run_name']

trained_run_dirs

In [ ]:
def pick_checkpoint(run_dir: Path) -> Path:
    candidates = [
        run_dir / 'checkpoint_best_total.pth',
        run_dir / 'checkpoint_best_regular.pth',
        run_dir / 'checkpoint_best_ema.pth',
    ]
    for c in candidates:
        if c.is_file():
            return c
    raise FileNotFoundError(f'No best checkpoint found in {run_dir}')


def load_coco_valid():
    ann_path = DATASET_DIR / 'valid' / '_annotations.coco.json'
    coco = json.loads(ann_path.read_text(encoding='utf-8'))
    imgs = coco.get('images', [])
    anns = coco.get('annotations', [])

    imgs_by_id = {int(i['id']): i for i in imgs}
    anns_by_img: Dict[int, List[Dict]] = {}
    for a in anns:
        anns_by_img.setdefault(int(a['image_id']), []).append(a)
    return imgs, imgs_by_id, anns_by_img


VAL_IMGS, VAL_IMGS_BY_ID, VAL_ANNS_BY_IMG = load_coco_valid()
VALID_IMG_DIR = DATASET_DIR / 'valid'
print('valid images:', len(VAL_IMGS))

In [ ]:
def ann_to_mask(ann: Dict, height: int, width: int) -> np.ndarray:
    seg = ann.get('segmentation', None)
    if not seg:
        return np.zeros((height, width), dtype=np.uint8)

    if isinstance(seg, list):
        mask = np.zeros((height, width), dtype=np.uint8)
        for poly in seg:
            pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2)
            cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
        return mask

    if isinstance(seg, dict):
        try:
            from pycocotools import mask as mask_utils
            return mask_utils.decode(seg).astype(np.uint8)
        except Exception:
            return np.zeros((height, width), dtype=np.uint8)

    return np.zeros((height, width), dtype=np.uint8)


def get_gt_masks_for_image(img_id: int) -> Tuple[List[np.ndarray], int, int]:
    info = VAL_IMGS_BY_ID[int(img_id)]
    h, w = int(info['height']), int(info['width'])
    anns = VAL_ANNS_BY_IMG.get(int(img_id), [])
    masks = [ann_to_mask(a, h, w) for a in anns]
    return masks, h, w


def get_pred_masks_and_conf(det) -> Tuple[List[np.ndarray], List[float]]:
    masks: List[np.ndarray] = []
    confs: List[float] = []

    # RF-DETR prediction object can expose one or many masks depending on version.
    pred_mask = getattr(det, 'mask', None)
    pred_conf = getattr(det, 'confidence', None)

    if pred_mask is not None:
        arr = np.asarray(pred_mask)
        if arr.ndim == 2:
            masks.append((arr > 0).astype(np.uint8))
        elif arr.ndim == 3:
            for m in arr:
                masks.append((m > 0).astype(np.uint8))

    if pred_conf is not None:
        c = np.asarray(pred_conf).reshape(-1)
        confs = [float(x) for x in c]

    if len(confs) != len(masks):
        confs = [1.0] * len(masks)

    return masks, confs


def iou_binary(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0


def greedy_match(pred_masks: List[np.ndarray], gt_masks: List[np.ndarray], iou_thr: float = 0.5):
    candidates: List[Tuple[float, int, int]] = []
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            score = iou_binary(pm, gm)
            if score >= iou_thr:
                candidates.append((score, i, j))
    candidates.sort(key=lambda x: x[0], reverse=True)

    used_p, used_g = set(), set()
    pairs = []
    for score, i, j in candidates:
        if i in used_p or j in used_g:
            continue
        used_p.add(i)
        used_g.add(j)
        pairs.append((i, j, score))
    return pairs, used_p, used_g

In [ ]:
def eval_run_at_conf(infer_model: RFDETRSegSmall, conf_thr: float) -> Dict[str, float]:
    tp = fp = fn = 0
    ious = []
    gt_total_area = 0
    pred_total_area = 0

    for info in VAL_IMGS:
        img_id = int(info['id'])
        img_name = info['file_name']
        img_path = VALID_IMG_DIR / img_name

        gt_masks, h, w = get_gt_masks_for_image(img_id)
        gt_total_area += sum(int(gm.sum()) for gm in gt_masks)

        det = infer_model.predict(str(img_path), threshold=float(conf_thr))
        pred_masks, pred_confs = get_pred_masks_and_conf(det)
        pred_masks = [m for m, c in zip(pred_masks, pred_confs) if c >= conf_thr]

        resized_preds = [cv2.resize(pm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) for pm in pred_masks]
        pred_masks = [(pm > 0).astype(np.uint8) for pm in resized_preds]
        pred_total_area += sum(int(pm.sum()) for pm in pred_masks)

        pairs, used_p, used_g = greedy_match(pred_masks, gt_masks, iou_thr=IOU_THRESHOLD)
        tp += len(pairs)
        fp += len(pred_masks) - len(used_p)
        fn += len(gt_masks) - len(used_g)
        ious.extend([p[2] for p in pairs])

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    mean_iou = float(np.mean(ious)) if ious else 0.0
    area_rel_error = abs(pred_total_area - gt_total_area) / max(gt_total_area, 1)
    score = (0.6 * f1) + (0.3 * mean_iou) + (0.1 * (1.0 - area_rel_error))

    return {
        'conf': float(conf_thr),
        'tp': int(tp),
        'fp': int(fp),
        'fn': int(fn),
        'precision': float(precision),
        'recall': float(recall),
        'f1': float(f1),
        'mean_iou': float(mean_iou),
        'gt_total_area_px': int(gt_total_area),
        'pred_total_area_px': int(pred_total_area),
        'area_rel_error': float(area_rel_error),
        'score': float(score),
    }

In [ ]:
run_summaries = []
all_conf_rows = []

for cfg in RUN_MATRIX:
    run_name = cfg['run_name']
    run_dir = trained_run_dirs[run_name]
    ckpt = pick_checkpoint(run_dir)

    print(f'\n=== Confidence sweep: {run_name} ===')
    infer_model = RFDETRSegSmall(pretrain_weights=str(ckpt))

    rows = []
    for conf_thr in CONF_CANDIDATES:
        row = eval_run_at_conf(infer_model, conf_thr=conf_thr)
        row['run_name'] = run_name
        rows.append(row)
        all_conf_rows.append(row)

    sweep_df = pd.DataFrame(rows).sort_values('score', ascending=False).reset_index(drop=True)
    sweep_csv = BENCH_ROOT / f'validation_conf_sweep_{run_name}.csv'
    sweep_df.to_csv(sweep_csv, index=False)

    best = sweep_df.iloc[0].to_dict()
    best['checkpoint'] = str(ckpt)
    best['sweep_csv'] = str(sweep_csv)
    run_summaries.append(best)

    print('best conf:', best['conf'], 'score:', round(float(best['score']), 4))

runs_df = pd.DataFrame(run_summaries).sort_values('score', ascending=False).reset_index(drop=True)
runs_df.to_csv(BENCH_ROOT / 'run_ranking.csv', index=False)
runs_df.head(len(runs_df))

In [ ]:
if len(runs_df) == 0:
    raise RuntimeError('No run summaries found.')

CHAMP = runs_df.iloc[0].to_dict()
CHAMP_RUN = CHAMP['run_name']
CHAMP_CONF = float(CHAMP['conf'])
CHAMP_CKPT = Path(CHAMP['checkpoint'])

print('Champion run: ', CHAMP_RUN)
print('Champion conf:', CHAMP_CONF)
print('Checkpoint:   ', CHAMP_CKPT)

(BENCH_ROOT / 'champion_run.json').write_text(
    json.dumps(
        {
            'run_name': CHAMP_RUN,
            'best_conf': CHAMP_CONF,
            'checkpoint': str(CHAMP_CKPT),
            'score': float(CHAMP['score']),
        },
        indent=2,
    ),
    encoding='utf-8',
)

In [ ]:
def evaluate_and_export_reports(run_name: str, ckpt_path: Path, conf_thr: float):
    infer_model = RFDETRSegSmall(pretrain_weights=str(ckpt_path))

    run_report_dir = REPORTS_ROOT / run_name
    run_viz_dir = VIZ_ROOT / run_name
    run_report_dir.mkdir(parents=True, exist_ok=True)
    run_viz_dir.mkdir(parents=True, exist_ok=True)

    instance_rows = []
    totals_rows = []

    for info in VAL_IMGS:
        img_id = int(info['id'])
        img_name = info['file_name']
        img_path = VALID_IMG_DIR / img_name

        gt_masks, h, w = get_gt_masks_for_image(img_id)
        det = infer_model.predict(str(img_path), threshold=float(conf_thr))
        pred_masks, pred_confs = get_pred_masks_and_conf(det)
        pred_masks = [m for m, c in zip(pred_masks, pred_confs) if c >= conf_thr]
        pred_masks = [(cv2.resize(pm.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST) > 0).astype(np.uint8) for pm in pred_masks]

        pairs, used_p, used_g = greedy_match(pred_masks, gt_masks, iou_thr=IOU_THRESHOLD)

        gt_area_total = int(sum(int(gm.sum()) for gm in gt_masks))
        pred_area_total = int(sum(int(pm.sum()) for pm in pred_masks))

        totals_rows.append(
            {
                'Image': img_name,
                'GT_Count': len(gt_masks),
                'AI_Count': len(pred_masks),
                'Matched_Count': len(pairs),
                'FP_Count': len(pred_masks) - len(pairs),
                'FN_Count': len(gt_masks) - len(pairs),
                'GT_Area_px': gt_area_total,
                'AI_Area_px': pred_area_total,
                'GT_Area_um2': round(gt_area_total * AREA_FACTOR, 4),
                'AI_Area_um2': round(pred_area_total * AREA_FACTOR, 4),
                'Conf': conf_thr,
                'Run': run_name,
            }
        )

        for p_idx, g_idx, score in pairs:
            p_area = int(pred_masks[p_idx].sum())
            g_area = int(gt_masks[g_idx].sum())
            instance_rows.append(
                {
                    'Image': img_name,
                    'Match_Type': 'TP',
                    'AI_Index': p_idx,
                    'GT_Index': g_idx,
                    'IoU': round(score, 6),
                    'GT_Area_px': g_area,
                    'AI_Area_px': p_area,
                    'GT_Area_um2': round(g_area * AREA_FACTOR, 4),
                    'AI_Area_um2': round(p_area * AREA_FACTOR, 4),
                    'Conf': conf_thr,
                    'Run': run_name,
                }
            )

        for p_idx, pm in enumerate(pred_masks):
            if p_idx in used_p:
                continue
            p_area = int(pm.sum())
            instance_rows.append(
                {
                    'Image': img_name,
                    'Match_Type': 'FP',
                    'AI_Index': p_idx,
                    'GT_Index': -1,
                    'IoU': 0.0,
                    'GT_Area_px': 0,
                    'AI_Area_px': p_area,
                    'GT_Area_um2': 0.0,
                    'AI_Area_um2': round(p_area * AREA_FACTOR, 4),
                    'Conf': conf_thr,
                    'Run': run_name,
                }
            )

        for g_idx, gm in enumerate(gt_masks):
            if g_idx in used_g:
                continue
            g_area = int(gm.sum())
            instance_rows.append(
                {
                    'Image': img_name,
                    'Match_Type': 'FN',
                    'AI_Index': -1,
                    'GT_Index': g_idx,
                    'IoU': 0.0,
                    'GT_Area_px': g_area,
                    'AI_Area_px': 0,
                    'GT_Area_um2': round(g_area * AREA_FACTOR, 4),
                    'AI_Area_um2': 0.0,
                    'Conf': conf_thr,
                    'Run': run_name,
                }
            )

        # IoU viz
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        gt_combined = np.zeros((h, w), dtype=np.uint8)
        pred_combined = np.zeros((h, w), dtype=np.uint8)
        for gm in gt_masks:
            gt_combined = np.logical_or(gt_combined, gm).astype(np.uint8)
        for pm in pred_masks:
            pred_combined = np.logical_or(pred_combined, pm).astype(np.uint8)

        inter = np.logical_and(gt_combined, pred_combined).sum()
        union = np.logical_or(gt_combined, pred_combined).sum()
        img_iou = (inter / union) if union > 0 else 0.0

        overlay = img.copy()
        overlay[gt_combined == 1] = (overlay[gt_combined == 1] * 0.5 + np.array([0, 255, 0]) * 0.5).astype(np.uint8)
        overlay[pred_combined == 1] = (overlay[pred_combined == 1] * 0.5 + np.array([255, 0, 0]) * 0.5).astype(np.uint8)
        overlap = np.logical_and(gt_combined, pred_combined)
        overlay[overlap] = (overlay[overlap] * 0.5 + np.array([255, 255, 0]) * 0.5).astype(np.uint8)

        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        fig.suptitle(f"{img_name} | IoU={img_iou:.3f} | GT={len(gt_masks)} | Pred={len(pred_masks)}")
        axes[0].imshow(img); axes[0].set_title('Original'); axes[0].axis('off')
        axes[1].imshow(overlay); axes[1].set_title('GT=green AI=red overlap=yellow'); axes[1].axis('off')
        plt.savefig(run_viz_dir / f"iou_viz_{Path(img_name).stem}.png", dpi=140, bbox_inches='tight')
        plt.close(fig)

    instance_csv = run_report_dir / 'placenta_instance_report_rfdetr.csv'
    totals_csv = run_report_dir / 'placenta_totals_report_rfdetr.csv'
    pd.DataFrame(instance_rows).to_csv(instance_csv, index=False)
    pd.DataFrame(totals_rows).to_csv(totals_csv, index=False)

    return instance_csv, totals_csv, run_viz_dir


inst_csv, totals_csv, viz_dir = evaluate_and_export_reports(CHAMP_RUN, CHAMP_CKPT, CHAMP_CONF)
print('Instance report:', inst_csv)
print('Totals report:  ', totals_csv)
print('Viz dir:        ', viz_dir)

In [ ]:
def benchmark_inference(run_name: str, ckpt_path: Path, conf_thr: float, repeats: int = 3, warmup: int = 1) -> Path:
    infer_model = RFDETRSegSmall(pretrain_weights=str(ckpt_path))

    # Try inference optimization when available.
    if hasattr(infer_model, 'optimize_for_inference'):
        try:
            infer_model.optimize_for_inference()
        except Exception as e:
            print(f'optimize_for_inference skipped: {e}')

    img_paths = [VALID_IMG_DIR / i['file_name'] for i in VAL_IMGS]

    # Warmup
    for _ in range(warmup):
        for p in img_paths:
            _ = infer_model.predict(str(p), threshold=float(conf_thr))

    timings = []
    peak_mem_mb = None

    for _ in range(repeats):
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()
            torch.cuda.synchronize()

        t0 = time.time()
        n_preds = 0
        for p in img_paths:
            _ = infer_model.predict(str(p), threshold=float(conf_thr))
            n_preds += 1

        if torch.cuda.is_available():
            torch.cuda.synchronize()
            peak_mem_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

        elapsed = time.time() - t0
        timings.append(elapsed)

    mean_elapsed = float(np.mean(timings))
    std_elapsed = float(np.std(timings))
    ips = (len(img_paths) / mean_elapsed) if mean_elapsed > 0 else 0.0

    payload = {
        'run_name': run_name,
        'checkpoint': str(ckpt_path),
        'conf': float(conf_thr),
        'n_images': len(img_paths),
        'repeats': repeats,
        'warmup': warmup,
        'elapsed_sec_mean': mean_elapsed,
        'elapsed_sec_std': std_elapsed,
        'images_per_sec': ips,
        'peak_gpu_mem_mb': peak_mem_mb,
    }

    out_json = BENCH_ROOT / f'inference_benchmark_{run_name}.json'
    out_json.write_text(json.dumps(payload, indent=2), encoding='utf-8')
    return out_json


bench_json = benchmark_inference(CHAMP_RUN, CHAMP_CKPT, CHAMP_CONF, repeats=3, warmup=1)
print('Benchmark:', bench_json)

In [ ]:
final_summary = {
    'champion_run': CHAMP_RUN,
    'champion_conf': CHAMP_CONF,
    'champion_checkpoint': str(CHAMP_CKPT),
    'run_ranking_csv': str(BENCH_ROOT / 'run_ranking.csv'),
    'instance_report_csv': str(inst_csv),
    'totals_report_csv': str(totals_csv),
    'iou_viz_dir': str(viz_dir),
    'inference_benchmark_json': str(bench_json),
}

final_summary_path = BENCH_ROOT / 'final_summary.json'
final_summary_path.write_text(json.dumps(final_summary, indent=2), encoding='utf-8')
print(json.dumps(final_summary, indent=2))
print('Saved:', final_summary_path)